# Kaggle GPU Runner — Track B 3D CNN (Telup Dataset, 120 Frames)

Self-contained notebook for Kaggle **GPU T4**.
- Auto-discovers `.npy` face clips and `*kinetics*.pth` checkpoints under `/kaggle/input`.
- Trains `3D ResNet` (ResNet-18), `3D WideResNet` (WideResNet-50), and `3D ResNeXt` (ResNeXt-101) (5-Fold GroupKFold + Hold-out Test).
- Profiles `Params`, `GFLOPs`, `Size_KB`, `Latency_ms`, generates English figures & Top-K 3D Score-CAM.
- Saves results in the exact JSON/checkpoint format of `main_telup_point_vs_pixel.ipynb` and zips them for download.


In [ ]:
import json, math, os, random, re, shutil, time
from collections import OrderedDict
from functools import partial
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GroupKFold
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.utils.flop_counter import FlopCounterMode

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# ============================================================================
# 1. AUTO-DISCOVER KAGGLE INPUTS & CONFIG
# ============================================================================
INPUT_ROOT = Path('/kaggle/input')
NPY_FILES = sorted(INPUT_ROOT.rglob('*.npy'))
PTH_FILES = {p.name: p for p in INPUT_ROOT.rglob('*.pth')}

print(f'Found {len(NPY_FILES)} .npy clips under {INPUT_ROOT}')
print('Found .pth checkpoints:', list(PTH_FILES.keys()))
assert len(NPY_FILES) > 0, 'No .npy files found in /kaggle/input! Check your attached dataset.'

OUT_ROOT = Path('/kaggle/working/results_telup_track_b')
CKPT_DIR = OUT_ROOT / 'checkpoints' / 'track_b'
PER_MODEL_DIR = OUT_ROOT / 'tables' / 'per_model'
FIG_CURVES_DIR = OUT_ROOT / 'figures' / 'training_curves'
FIG_CAM_DIR = OUT_ROOT / 'figures' / 'xai_scorecam'
for d in [CKPT_DIR, PER_MODEL_DIR, FIG_CURVES_DIR, FIG_CAM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CFG = {
    'seq_len': 120,
    'sample_size': 112,
    'num_classes': 2,
    'folds': 5,
    'seed': 42,
    'test_subjects': [f'subject_{i:02d}' for i in range(16, 21)],
    'kinetics_mean': (114.7748, 107.7354, 99.4750),
    'finetune_mode': 'last_block',
    'batch_size': 4,
    'epochs': 20,
    'warmup_epochs': 2,
    'lr': 1e-4,
    'min_lr': 1e-6,
    'weight_decay': 1e-2,
    'grad_clip': 1.0,
    'label_smoothing': 0.10,
    'patience': 7,
    'scorecam_top_k': 32,
}

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG['seed'])


In [ ]:
# ============================================================================
# 2. DATASET & SUBJECT-INDEPENDENT SPLIT
# ============================================================================
records = []
for npy_path in NPY_FILES:
    fname = npy_path.name
    label = 0 if fname.startswith('non-drowsiness') else 1
    m = re.search(r'(subject_\d+)', fname)
    subject = m.group(1) if m else 'unknown'
    records.append({'path': str(npy_path), 'filename': fname, 'subject': subject, 'label': label})

df_px = pd.DataFrame(records)
df_train_pool = df_px[~df_px['subject'].isin(CFG['test_subjects'])].reset_index(drop=True)
df_test_holdout = df_px[df_px['subject'].isin(CFG['test_subjects'])].reset_index(drop=True)
print(f'Train Pool: {len(df_train_pool)} | Hold-out Test: {len(df_test_holdout)}')

class PixelClipDataset(Dataset):
    def __init__(self, paths, labels, augment=False):
        self.paths = list(paths)
        self.labels = list(labels)
        self.augment = augment
        self.mean = torch.tensor(CFG['kinetics_mean'], dtype=torch.float32).view(3, 1, 1, 1)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        clip = np.load(self.paths[idx])
        x = torch.from_numpy(clip).permute(3, 0, 1, 2).float()
        if self.augment:
            if torch.rand(1).item() < 0.5:
                x = torch.flip(x, dims=[3])
            x = torch.clamp(x * torch.empty(1).uniform_(0.9, 1.1).item(), 0.0, 255.0)
        return x - self.mean, torch.tensor(self.labels[idx], dtype=torch.long)


In [ ]:
# ============================================================================
# 3. 3D CNN ARCHITECTURES (RESNET, WIDERESNET, RESNEXT, DENSENET)
# ============================================================================
def _downsample_basic_block(x, planes, stride):
    out = F.avg_pool3d(x, kernel_size=1, stride=stride)
    zero_pads = torch.zeros(out.size(0), planes - out.size(1), out.size(2), out.size(3), out.size(4), device=out.device, dtype=out.dtype)
    return torch.cat([out, zero_pads], dim=1)

class BasicBlock3D(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv3d(inplanes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv3d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(planes)
        self.downsample = downsample
    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        out = self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x)))))
        return self.relu(out + res)

class WideBottleneck(nn.Module):
    expansion = 2
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv3d(inplanes, planes, 1, bias=False); self.bn1 = nn.BatchNorm3d(planes)
        self.conv2 = nn.Conv3d(planes, planes, 3, stride=stride, padding=1, bias=False); self.bn2 = nn.BatchNorm3d(planes)
        self.conv3 = nn.Conv3d(planes, planes * 2, 1, bias=False); self.bn3 = nn.BatchNorm3d(planes * 2)
        self.relu = nn.ReLU(inplace=True); self.downsample = downsample
    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        out = self.bn3(self.conv3(self.relu(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))))))
        return self.relu(out + res)

class ResNeXtBottleneck(nn.Module):
    expansion = 2
    def __init__(self, inplanes, planes, cardinality, stride=1, downsample=None):
        super().__init__()
        mid = cardinality * int(planes / 32)
        self.conv1 = nn.Conv3d(inplanes, mid, 1, bias=False); self.bn1 = nn.BatchNorm3d(mid)
        self.conv2 = nn.Conv3d(mid, mid, 3, stride=stride, padding=1, groups=cardinality, bias=False); self.bn2 = nn.BatchNorm3d(mid)
        self.conv3 = nn.Conv3d(mid, planes * 2, 1, bias=False); self.bn3 = nn.BatchNorm3d(planes * 2)
        self.relu = nn.ReLU(inplace=True); self.downsample = downsample
    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        out = self.bn3(self.conv3(self.relu(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))))))
        return self.relu(out + res)

class BaseResNetFamily3D(nn.Module):
    def __init__(self, block, layers, planes_list, sample_size=112, sample_duration=120, shortcut_type='B', num_classes=2, cardinality=None):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv3d(3, 64, 7, stride=(1, 2, 2), padding=(3, 3, 3), bias=False)
        self.bn1 = nn.BatchNorm3d(64); self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, planes_list[0], layers[0], shortcut_type, 1, cardinality)
        self.layer2 = self._make_layer(block, planes_list[1], layers[1], shortcut_type, 2, cardinality)
        self.layer3 = self._make_layer(block, planes_list[2], layers[2], shortcut_type, 2, cardinality)
        self.layer4 = self._make_layer(block, planes_list[3], layers[3], shortcut_type, 2, cardinality)
        self.avgpool = nn.AvgPool3d((math.ceil(sample_duration / 16), math.ceil(sample_size / 32), math.ceil(sample_size / 32)), stride=1)
        fc_in = (cardinality * 32 * block.expansion) if cardinality else (planes_list[3] * block.expansion)
        self.fc = nn.Linear(fc_in, num_classes)
    def _make_layer(self, block, planes, blocks, shortcut_type, stride=1, cardinality=None):
        ds = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            if shortcut_type == 'A':
                ds = partial(_downsample_basic_block, planes=planes * block.expansion, stride=stride)
            else:
                ds = nn.Sequential(nn.Conv3d(self.inplanes, planes * block.expansion, 1, stride=stride, bias=False), nn.BatchNorm3d(planes * block.expansion))
        args = (self.inplanes, planes, cardinality, stride, ds) if cardinality else (self.inplanes, planes, stride, ds)
        layers = [block(*args)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, cardinality) if cardinality else block(self.inplanes, planes))
        return nn.Sequential(*layers)
    def forward_features(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        return self.layer4(self.layer3(self.layer2(self.layer1(x))))
    def forward(self, x):
        return self.fc(self.avgpool(self.forward_features(x)).flatten(1))

CKPT_MAP = {
    '3D ResNet': 'resnet-18-kinetics.pth',
    '3D WideResNet': 'wideresnet-50-kinetics.pth',
    '3D ResNeXt': 'resnext-101-kinetics.pth',
}

def build_pixel_model(model_name, load_pretrained=True):
    s, t, nc = CFG['sample_size'], CFG['seq_len'], CFG['num_classes']
    builders = {
        '3D ResNet': lambda: BaseResNetFamily3D(BasicBlock3D, [2, 2, 2, 2], [64, 128, 256, 512], s, t, 'A', nc),
        '3D WideResNet': lambda: BaseResNetFamily3D(WideBottleneck, [3, 4, 6, 3], [128, 256, 512, 1024], s, t, 'B', nc),
        '3D ResNeXt': lambda: BaseResNetFamily3D(ResNeXtBottleneck, [3, 4, 23, 3], [128, 256, 512, 1024], s, t, 'B', nc, cardinality=32),
    }
    model = builders[model_name]()
    ckpt_name = CKPT_MAP[model_name]
    effective_mode = CFG['finetune_mode']
    if load_pretrained and ckpt_name in PTH_FILES:
        ckpt = torch.load(PTH_FILES[ckpt_name], map_location='cpu', weights_only=False)
        state = ckpt.get('state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
        ms, cleaned = model.state_dict(), {}
        for k, v in state.items():
            name = k[7:] if k.startswith('module.') else k
            if not name.startswith(('fc.', 'classifier.')) and name in ms and ms[name].shape == v.shape:
                cleaned[name] = v
        model.load_state_dict(cleaned, strict=False)
        print(f'[PRETRAINED] {model_name}: loaded {len(cleaned)}/{len(ms)} tensors from {ckpt_name}')
    elif load_pretrained:
        print(f'[WARNING] {ckpt_name} not found in /kaggle/input! Switching to full training.')
        effective_mode = 'full'

    if effective_mode == 'last_block':
        for p in model.parameters(): p.requires_grad = False
        for n, p in model.named_parameters():
            if n.startswith(('layer4.', 'fc.')): p.requires_grad = True
    return model


In [ ]:
# ============================================================================
# 4. TRAINING, EVALUATION, BENCHMARK & TOP-K 3D SCORE-CAM ENGINE
# ============================================================================
def safe_slug(name):
    return name.replace(' ', '_').replace('+', '_')

def profile_model(model, model_name, num_runs=30, warmup=5):
    model = model.to(DEVICE).eval()
    total_p = sum(p.numel() for p in model.parameters())
    train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    dummy = torch.randn(1, 3, CFG['seq_len'], CFG['sample_size'], CFG['sample_size'], device=DEVICE)
    with torch.no_grad():
        fc = FlopCounterMode(display=False)
        with fc: _ = model(dummy)
        gflops = fc.get_total_flops() / 1e9
        for _ in range(warmup): _ = model(dummy)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(num_runs): _ = model(dummy)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        lat_ms = ((time.perf_counter() - t0) / num_runs) * 1000.0
    return {'Model': model_name, 'Total_Params': int(total_p), 'Trainable_Params': int(train_p),
            'GFLOPs': round(float(gflops), 6), 'Size_KB': round((total_p * 4) / 1024.0, 2), 'Latency_ms': round(float(lat_ms), 3)}

@torch.no_grad()
def _plot_scorecam_row(model, model_name, row, out_path):
    ds = PixelClipDataset([row['path']], [row['label']], augment=False)
    x_t, lbl = ds[0]
    x_in = x_t.unsqueeze(0).to(DEVICE)
    prob = float(torch.softmax(model(x_in), dim=1)[0, int(lbl)].item())
    fmap = model.forward_features(x_in)
    k = min(CFG['scorecam_top_k'], fmap.size(1))
    top_idx = torch.topk(fmap.abs().mean(dim=(0, 2, 3, 4)), k=k).indices
    masks = []
    for ch in top_idx:
        m_up = F.interpolate(fmap[:, ch:ch+1], size=(CFG['seq_len'], CFG['sample_size'], CFG['sample_size']), mode='trilinear', align_corners=False)
        mn, mx = m_up.min(), m_up.max()
        masks.append((m_up - mn) / (mx - mn) if (mx - mn) > 1e-6 else torch.zeros_like(m_up))
    masks_t = torch.cat(masks, dim=0)
    weights = torch.cat([torch.softmax(model(x_in * masks_t[s:s+8]), dim=1)[:, int(lbl)] for s in range(0, k, 8)], dim=0).view(k, 1, 1, 1, 1)
    cam = F.relu((weights * masks_t).sum(dim=0)).squeeze().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-6)
    raw = np.load(row['path'])
    key_frames = np.linspace(0, CFG['seq_len'] - 1, 6, dtype=int)
    fig, axes = plt.subplots(1, 6, figsize=(15, 3.1), layout='constrained')
    for ax, f_idx in zip(axes, key_frames):
        hm = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * cam[f_idx]), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
        ax.imshow(cv2.addWeighted(raw[f_idx], 0.58, hm, 0.42, 0)); ax.set_title(f'Frame {f_idx+1}', fontsize=9); ax.axis('off')
    lbl_s = 'Drowsy (1)' if int(lbl) == 1 else 'Non-Drowsy (0)'
    fig.suptitle(f"3D Score-CAM ({model_name}) — {row['subject']} | {row['filename']} | True: {lbl_s} | Prob: {prob*100:.1f}%", fontweight='bold')
    fig.savefig(out_path, dpi=180); plt.show(); plt.close(fig)

@torch.no_grad()
def run_scorecam(model_name, ckpt_path, extra_subjects=('subject_01', 'subject_13')):
    model = build_pixel_model(model_name, load_pretrained=False).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict']); model.eval()
    for tag, lbl_val in [('drowsy', 1), ('nondrowsy', 0)]:
        sub = df_test_holdout[df_test_holdout['label'] == lbl_val]
        if not sub.empty:
            _plot_scorecam_row(model, model_name, sub.iloc[0], FIG_CAM_DIR / f'{safe_slug(model_name)}_scorecam_holdout_{tag}.png')
    for subj in extra_subjects:
        for tag, lbl_val in [('drowsy', 1), ('nondrowsy', 0)]:
            sub = df_px[(df_px['subject'] == subj) & (df_px['label'] == lbl_val)]
            if not sub.empty:
                _plot_scorecam_row(model, model_name, sub.iloc[0], FIG_CAM_DIR / f'{safe_slug(model_name)}_scorecam_{subj}_{tag}.png')

def run_pixel_model(model_name):
    gkf = GroupKFold(n_splits=CFG['folds'])
    fold_metrics, histories, all_t, all_p = [], [], [], []
    crit = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])
    print(f'\n==================== 5-FOLD CV: {model_name} ====================')
    for fold, (tr_i, val_i) in enumerate(gkf.split(df_train_pool, df_train_pool['label'], groups=df_train_pool['subject']), start=1):
        seed_everything(CFG['seed'] + fold)
        tr_df, val_df = df_train_pool.iloc[tr_i].reset_index(drop=True), df_train_pool.iloc[val_i].reset_index(drop=True)
        tr_loader = DataLoader(PixelClipDataset(tr_df['path'], tr_df['label'], True), batch_size=CFG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
        val_loader = DataLoader(PixelClipDataset(val_df['path'], val_df['label'], False), batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
        model = build_pixel_model(model_name, load_pretrained=True).to(DEVICE)
        decay = [p for p in model.parameters() if p.requires_grad and p.ndim >= 2]
        no_decay = [p for p in model.parameters() if p.requires_grad and p.ndim < 2]
        opt = optim.AdamW([{'params': decay, 'weight_decay': CFG['weight_decay']}, {'params': no_decay, 'weight_decay': 0.0}], lr=CFG['lr'])
        sched = optim.lr_scheduler.SequentialLR(opt, [
            optim.lr_scheduler.LinearLR(opt, 0.2, 1.0, total_iters=CFG['warmup_epochs']),
            optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['epochs'] - CFG['warmup_epochs'], eta_min=CFG['min_lr'])
        ], milestones=[CFG['warmup_epochs']])
        hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
        best_vl, best_va, best_p, best_t, pat = float('inf'), 0.0, [], [], 0
        ckpt_path = CKPT_DIR / f'{safe_slug(model_name)}_fold{fold}_best.pth'
        t_fold = time.time()
        for ep in range(1, CFG['epochs'] + 1):
            model.train(); tl = tc = tt = 0
            for xb, yb in tr_loader:
                xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
                opt.zero_grad(); logits = model(xb); loss = crit(logits, yb)
                loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip']); opt.step()
                tl += loss.item() * yb.size(0); tc += (logits.argmax(1) == yb).sum().item(); tt += yb.size(0)
            sched.step()
            model.eval(); vl = vc = vt = 0; fp, ft = [], []
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
                    logits = model(xb); vl += crit(logits, yb).item() * yb.size(0)
                    pr = logits.argmax(1); vc += (pr == yb).sum().item(); vt += yb.size(0)
                    fp.extend(pr.cpu().numpy()); ft.extend(yb.cpu().numpy())
            etl, evl, eta, eva = tl/tt, vl/vt, tc/tt, vc/vt
            hist['train_loss'].append(etl); hist['val_loss'].append(evl); hist['train_acc'].append(eta); hist['val_acc'].append(eva)
            if (evl < best_vl - 1e-4) or (abs(evl - best_vl) <= 1e-4 and eva >= best_va):
                best_vl, best_va, best_p, best_t, pat = evl, eva, fp.copy(), ft.copy(), 0
                torch.save({'model_state_dict': model.state_dict(), 'best_val_loss': best_vl, 'best_val_acc': best_va, 'epoch': ep}, ckpt_path)
            else:
                pat += 1
                if pat >= CFG['patience']:
                    for _ in range(CFG['epochs'] - ep):
                        hist['train_loss'].append(etl); hist['val_loss'].append(best_vl); hist['train_acc'].append(eta); hist['val_acc'].append(best_va)
                    break
        fold_metrics.append({'fold': fold, 'val_loss': float(best_vl), 'accuracy': float(accuracy_score(best_t, best_p)),
                             'precision': float(precision_score(best_t, best_p, zero_division=0)),
                             'recall': float(recall_score(best_t, best_p, zero_division=0)),
                             'f1': float(f1_score(best_t, best_p, zero_division=0)), 'ckpt_path': str(ckpt_path)})
        histories.append(hist); all_t.extend(best_t); all_p.extend(best_p)
        print(f'  Fold {fold} ({time.time()-t_fold:.1f}s) -> Val Loss: {best_vl:.4f} | Acc: {fold_metrics[-1]["accuracy"]*100:.2f}% | F1: {fold_metrics[-1]["f1"]*100:.2f}%')

    # Hold-out Test Ensemble
    te_loader = DataLoader(PixelClipDataset(df_test_holdout['path'], df_test_holdout['label'], False), batch_size=CFG['batch_size'], shuffle=False)
    all_probs, y_true = [], None
    for fm in fold_metrics:
        m = build_pixel_model(model_name, load_pretrained=False).to(DEVICE)
        m.load_state_dict(torch.load(fm['ckpt_path'], map_location=DEVICE, weights_only=False)['model_state_dict']); m.eval()
        fp, lbls = [], []
        with torch.no_grad():
            for xb, yb in te_loader:
                fp.append(torch.softmax(m(xb.to(DEVICE)), dim=1).cpu().numpy()); lbls.extend(yb.numpy())
        all_probs.append(np.concatenate(fp, axis=0)); y_true = np.array(lbls)
    y_pred = np.argmax(np.mean(all_probs, axis=0), axis=1)

    df_m = pd.DataFrame(fold_metrics)
    res = {'track': 'Track B', 'model': model_name, 'fold_metrics': fold_metrics,
           'cv_val_loss': float(df_m['val_loss'].mean()), 'cv_acc_mean': float(df_m['accuracy'].mean()),
           'cv_acc_std': float(df_m['accuracy'].std()), 'cv_prec_mean': float(df_m['precision'].mean()),
           'cv_rec_mean': float(df_m['recall'].mean()), 'cv_f1_mean': float(df_m['f1'].mean()), 'cv_f1_std': float(df_m['f1'].std()),
           'test_acc': float(accuracy_score(y_true, y_pred)), 'test_prec': float(precision_score(y_true, y_pred, zero_division=0)),
           'test_rec': float(recall_score(y_true, y_pred, zero_division=0)), 'test_f1': float(f1_score(y_true, y_pred, zero_division=0))}
    eff = profile_model(build_pixel_model(model_name, load_pretrained=False), model_name)
    out_json = PER_MODEL_DIR / f'{safe_slug(model_name)}.json'
    out_json.write_text(json.dumps({**res, **eff}, indent=2), encoding='utf-8')
    print(f'  [Hold-Out Test] {model_name} -> Acc: {res["test_acc"]*100:.2f}% | F1: {res["test_f1"]*100:.2f}% | GFLOPs: {eff["GFLOPs"]}')

    # Plot CV curves
    ep_r = np.arange(1, CFG['epochs'] + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), layout='constrained')
    for arr, lbl, col, ls in [(np.array([h['train_loss'] for h in histories]), 'Train Loss', '#0072B2', '-'),
                              (np.array([h['val_loss'] for h in histories]), 'Val Loss', '#D55E00', '--')]:
        m, s = arr.mean(0), arr.std(0); axes[0].plot(ep_r, m, label=lbl, color=col, ls=ls, lw=2); axes[0].fill_between(ep_r, m-s, m+s, color=col, alpha=0.15)
    axes[0].set(title=f'{model_name} — 5-Fold Loss Curve', xlabel='Epoch', ylabel='Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    for arr, lbl, col, ls in [(np.array([h['train_acc'] for h in histories]), 'Train Acc', '#009E73', '-'),
                              (np.array([h['val_acc'] for h in histories]), 'Val Acc', '#E69F00', '--')]:
        m, s = arr.mean(0), arr.std(0); axes[1].plot(ep_r, m, label=lbl, color=col, ls=ls, lw=2); axes[1].fill_between(ep_r, m-s, m+s, color=col, alpha=0.15)
    axes[1].set(title=f'{model_name} — 5-Fold Accuracy Curve', xlabel='Epoch', ylabel='Accuracy', ylim=(0, 1.05)); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    sns.heatmap(confusion_matrix(all_t, all_p), annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[2],
                xticklabels=['Non-Drowsy (0)', 'Drowsy (1)'], yticklabels=['Non-Drowsy (0)', 'Drowsy (1)'])
    axes[2].set(title=f'{model_name} — CV Confusion Matrix', xlabel='Predicted', ylabel='True')
    fig.savefig(FIG_CURVES_DIR / f'{safe_slug(model_name)}_cv_curves.png', dpi=180); plt.show(); plt.close(fig)
    run_scorecam(model_name, fold_metrics[0]['ckpt_path'])
    return res, eff


## 5. Run Each 3D CNN Model (One Cell per Model)


In [ ]:
run_pixel_model('3D ResNet')


In [ ]:
run_pixel_model('3D WideResNet')


In [ ]:
run_pixel_model('3D ResNeXt')


## 5b. Extra Score-CAM Check (Subject 01 & Subject 13 from Saved Checkpoints)


In [ ]:
for m_name in ['3D ResNet', '3D WideResNet', '3D ResNeXt']:
    ckpt_p = CKPT_DIR / f'{safe_slug(m_name)}_fold1_best.pth'
    if ckpt_p.exists():
        run_scorecam(m_name, ckpt_p, extra_subjects=('subject_01', 'subject_13'))
    else:
        print(f'[SKIP] {m_name} checkpoint not found yet.')


## 6. Zip All Results for Download to Local PC


In [ ]:
zip_path = shutil.make_archive('/kaggle/working/results_telup_track_b', 'zip', OUT_ROOT)
print('Ready to download:', zip_path)
